In [2]:
%pip install -U langchain langchain-classic langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage, SystemMessage,HumanMessage
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from sentence_transformers import SentenceTransformer


load_dotenv()


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16288\3326550126.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
d:\python\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [4]:
%pip show langchain langchain-core langchain-classic langsmith

Name: langchainNote: you may need to restart the kernel to use updated packages.

Version: 1.4.7
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: d:\python\venv\lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-core
Version: 1.6.13
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: d:\python\venv\lib\site-packages
Requires: httpx, jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-chroma, langchain-classic, langchain-community, langchain-google-genai, langchain-groq, langchain-huggingface, langchain-ollama, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, langserve
---
Name: langchain-class

In [5]:
import langsmith
print("LangSmith version:", langsmith.__version__)
print("LangSmith path:", langsmith.__file__)

LangSmith version: 0.14.12
LangSmith path: d:\python\venv\lib\site-packages\langsmith\__init__.py


In [6]:
%pip uninstall -y langsmith
%pip install --no-cache-dir langsmith

Found existing installation: langsmith 0.14.12
Uninstalling langsmith-0.14.12:
  Successfully uninstalled langsmith-0.14.12
Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/883.6 kB ? eta -:--:--
   ---------------------------------------- 883.6/883.6 kB 38.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
llm = ChatGroq(model="openai/gpt-oss-120b")
# llm = ChatGroq(model="llama-3.1-8b-instant")

web_base_loader = WebBaseLoader('https://www.flipkart.com/apple-iphone-18-pro-glacier-256-gb/p/itm3874a83c12f72?pid=MOBHR2GUJEXEBDCF&')

docs = web_base_loader.load()

# print(docs[0].page_content[:1000])
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)
chunk_data = text_splitter.split_documents(docs)

# print("Total chunks:", len(chunk_data))

# for i, chunk in enumerate(chunk_data):
#     print(f"\nChunk {i + 1}:")
#     print(chunk.page_content)

## Create the embedding and store it to vector store
# vector_store = Chroma.from_documents(documents=chunk_data, embeddings ='sentence-transformers/all-MiniLM-L6-v2')
vector_store = Chroma.from_documents(documents=chunk_data, embedding = HuggingFaceEmbeddings())


retriever= vector_store.as_retriever()
retriever

system_msg = ''' You are an helpful q&a Assistant.
Answer the query from the provided  context . If you dont know the answer directly say Thankyou and I dont know the answer. Keep the answer concise . {context}'''

prompt = ChatPromptTemplate.from_messages([('system',system_msg),('human','{input}')])

question_answer_chain = create_stuff_documents_chain(llm,prompt)

rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({'input':'What is the listed price of the Apple iPhone 18 Pro with 256 GB storage? '})
response['answer']






Loading weights: 100%|██████████| 199/199 [00:00<00:00, 262.94it/s]


'The listed price is **₹1,64,900**.'

In [18]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Prompt for rewriting the question using chat history
context_system = """
Given the chat history and the latest user question,
rewrite the question as a standalone question if needed.
Do not answer the question.
"""

context_prompt = ChatPromptTemplate.from_messages([
    ("system", context_system),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# History-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, context_prompt
)

# Prompt for answering from retrieved context
q_a_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are a helpful Q&A assistant.
    Answer using only the provided context.
    If you don't know the answer, say you don't know.

    Context:
    {context}
    """),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Create the answer chain
question_answer_chain_1 = create_stuff_documents_chain(
    llm, q_a_prompt
)

# Create the final RAG chain
rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain_1
)

chat_history = []

question = "What is the price of iPhone 18 Pro with 256 GB storage?"

output = rag_chain.invoke({
    "input": question,
    "chat_history": chat_history
})

print(output["answer"])

The iPhone 18 Pro with 256 GB storage is listed at **₹1,64,900**. After applying offers, it can be bought for **₹1,58,900**.
